# Self-tuning pipeline: HPO as a KFP step with a config flag

Your design: hyperparameter search is a **step of the pipeline**, not a
separate ritual. One flag in the config switches the mode:

- `hpo.enabled: false` -> `resolve_hyperparameters` passes the config's
  values through (lr, embedding_hid, train_iters) - the fast run you know.
- `hpo.enabled: true`  -> the same component launches a
  **HyperparameterTuningJob** (Vizier), waits for it, reads the best trial,
  and hands the winning values to training. One run: search -> train winner
  -> gates -> promotion, with full lineage.

```
verify -> RESOLVE HP (flag: passthrough | Vizier search) -> train -> ...as always
```

**Key design points:**
- ONE lightweight component with a Python `if` inside - not `dsl.If` branches
  (no output-merging machinery; the graph is identical in both modes, only the
  resolve step's duration differs: seconds vs hours).
- **HPO gets its own machine type** (`hpo.machine_type`, default n1-standard-8)
  independent of the pipeline's training machine - trials can run on beefier
  hardware (or GPU: add accelerator to worker_pool_specs; check your T4 quota
  first) while the confirmation training stays on the standard machine.
- **Methodology unchanged:** trials optimize `val_accuracy` (400 episodes);
  the winner is then trained fresh IN the pipeline and judged by the gates on
  TEST episodes the search never optimized against.
- **Lineage:** the experiment log now records `lr` and
  `hyperparameter_source` (`config` vs `hpo`) - every run says where its
  parameters came from.
- `lr` now flows through the whole chain (wrapper -> component -> pipeline) -
  it previously fell back to TrainConfig's default; with HPO in play that had
  to be fixed.

**Before running - one-time steps (can be combined with the hypertune rebuild):**
1. `pyproject.toml`: add `cloudml-hypertune` to dependencies
2. `scripts/`: updated `train_pipeline_entry.py` (now takes `--lr`) and new
   `scripts/hpo_train_entry.py`
3. **REBUILD the image** (slow this once - dependency layer changes), then:
   `docker run --rm --entrypoint python IMAGE -c "import hypertune; print('ok')"`
4. Updated `configs/pipeline_config.json` (new `hpo` section + `training.lr`)

**Verified offline (KFP 2.16.1 + GCPC 2.22.0):** the resolve pattern (bool
flag, Python-if, NamedTuple outputs feeding a container component) compiles;
the full graph shows verify -> resolve -> train with lr/train_iters/
embedding_hid flowing from resolve's outputs and the experiment log consuming
lr + source; the HyperparameterTuningJob construction was verified to the
generated proto earlier (rung 9's first form). NOT verified: the run.
**Cost warning:** `enable_hpo=true` makes the pipeline run take hours
(12 trials); the resolve node just waits - that's expected, not a hang.

## Setup

In [1]:
# %pip install kfp google-cloud-pipeline-components
# (restart kernel after installing)

## Load config

In [2]:
import json
from fsl.validate import validate_config, ConfigError

with open("configs/pipeline_config.json") as f:
    CFG = json.load(f)

# Fail fast: full structural + consistency check BEFORE anything costly runs.
# This is the guard that turns "placeholder discovered mid-run" and "parallel >
# max_trials discovered after submit" into a first-second error listing every
# problem at once. require_hpo=True because this notebook submits tuning.
try:
    validate_config(CFG, require_hpo=CFG.get("hpo", {}).get("enabled", False))
except ConfigError as e:
    print(e)
    raise   # stop here — do not compile or submit with a bad config

PROJECT = CFG["project"]
REGION  = CFG["region"]
BUCKET  = CFG["bucket"]
DATASET = CFG["dataset"]
REPO    = CFG["pipeline_template_repo"]
PIPELINE_YAML = CFG["pipeline_yaml_path"]
IMAGE   = CFG["training_image_uri"]


Config OK — project=dark-data-discovery, dataset=omniglot, 5-way 5-shot, HPO off


## Component 1: `verify_frozen_component`

In [3]:
from typing import NamedTuple
from kfp import dsl
from kfp.dsl import ContainerSpec, OutputPath, Output, Metrics, HTML


@dsl.component(base_image="python:3.10", packages_to_install=["google-cloud-storage"])
def verify_frozen_component(project: str, bucket: str, dataset: str) -> str:
    from google.cloud import storage
    import json as _json
    c = storage.Client(project=project)
    m = _json.loads(c.bucket(bucket).blob(f"raw/{dataset}/MANIFEST.json").download_as_bytes().decode("utf-8"))
    return f"VERIFIED sha={m['archive_sha256'][:12]}"

## Component 2: `train_container` (now takes `lr`)

In [4]:
IMAGE = CFG['training_image_uri']


@dsl.container_component
def train_container(
    project: str,
    bucket: str,
    dataset: str,
    n_way: int,
    k_shot: int,
    query: int,
    seed: int,
    train_iters: int,
    embedding_hid: int,
    lr: float,
    accuracy_mean: OutputPath(float),
    accuracy_ci95: OutputPath(float),
    accuracy_std: OutputPath(float),
    train_time: OutputPath(float),
    model_dir: OutputPath(str),
    data_sha: OutputPath(str),
):
    return ContainerSpec(
        image=IMAGE,
        command=["python", "scripts/train_pipeline_entry.py"],
        args=[
            "--project", project, "--bucket", bucket, "--dataset", dataset,
            "--n-way", n_way, "--k-shot", k_shot, "--query", query,
            "--seed", seed, "--train-iters", train_iters, "--embedding-hid", embedding_hid,
            "--lr", lr,
            "--accuracy-mean-output-path", accuracy_mean,
            "--accuracy-ci95-output-path", accuracy_ci95,
            "--accuracy-std-output-path", accuracy_std,
            "--train-time-output-path", train_time,
            "--model-dir-output-path", model_dir,
            "--data-sha-output-path", data_sha,
        ],
    )

## Component 3: `resolve_hyperparameters` (NEW - the flag lives here)

Python `if` inside: passthrough or a blocking Vizier search. Returns
`(lr, embedding_hid, train_iters, source)` either way.

In [5]:
# --- RESOLVE HYPERPARAMETERS (lekki): flaga hpo.enabled decyduje ---
# Rozszerzenia: (1) raport HTML z rankingiem triali, (2) zwyciezca zapisywany
# do GCS (hpo/best/latest.json + kopia z timestampem - historia append-only),
# (3) przy enabled=false parametry czytane AUTOMATYCZNIE z GCS latest.json
#     (params_source="auto"; fallback na config gdy pliku brak; "config" wymusza config).
@dsl.component(base_image="python:3.10",
               packages_to_install=["google-cloud-aiplatform", "google-cloud-storage"])
def resolve_hyperparameters(
    project: str, region: str, bucket: str, dataset: str,
    training_image: str, hpo_machine_type: str,
    hpo_accelerator_type: str, hpo_accelerator_count: int,
    enable_hpo: bool,
    params_source: str,
    default_lr: float, default_embedding_hid: int, default_train_iters: int,
    hpo_max_trials: int, hpo_parallel: int, hpo_val_episodes: int,
    seed: int, n_way: int, k_shot: int, query: int,
    report: Output[HTML],
) -> NamedTuple("HP", [("lr", float), ("embedding_hid", int), ("train_iters", int), ("source", str)]):
    import json
    import time
    from collections import namedtuple

    HP = namedtuple("HP", ["lr", "embedding_hid", "train_iters", "source"])

    def write_report(html: str) -> None:
        with open(report.path, "w") as f:
            f.write("<!DOCTYPE html><html><head><style>"
                    "body{font-family:-apple-system,sans-serif;margin:24px;color:#1a1a1a;max-width:820px}"
                    "table{border-collapse:collapse;margin:14px 0}"
                    "td,th{border:1px solid #ddd;padding:5px 10px;font-size:13px;text-align:left}"
                    ".win{background:#e8f5ec;font-weight:600}"
                    ".meta{font-size:12px;color:#888}.src{font-size:15px;font-weight:600;color:#4C72B0}"
                    "</style></head><body>" + html + "</body></html>")

    # ---------- passthrough ----------
    if not enable_hpo:
        source, lr, hid, iters = "config", default_lr, default_embedding_hid, default_train_iters
        gcs_note = ""
        if params_source == "auto":
            try:
                from google.cloud import storage
                blob = storage.Client(project=project).bucket(bucket).blob("hpo/best/latest.json")
                saved = json.loads(blob.download_as_bytes().decode("utf-8"))
                lr, hid, iters = float(saved["lr"]), int(saved["embedding_hid"]), int(saved["train_iters"])
                source = f"gcs:{saved.get('saved_at', '?')}"
                gcs_note = (f"val_accuracy at tuning time: {saved.get('val_accuracy', float('nan')):.4f}; "
                            f"tuning job: {saved.get('hpo_job', '?')}")
                print(f"Params from GCS latest.json ({source}): lr={lr}, hid={hid}, iters={iters}")
            except Exception as e:  # brak pliku = jeszcze nigdy nie stroylismy
                print(f"No saved best params in GCS ({type(e).__name__}) - falling back to config.")
        else:
            print("params_source='config' - using config values explicitly.")
        write_report(
            f"<h2>Hyperparameters - passthrough (HPO disabled)</h2>"
            f"<p class='src'>source: {source}</p>"
            f"<table><tr><th>param</th><th>value</th></tr>"
            f"<tr><td>lr</td><td>{lr}</td></tr>"
            f"<tr><td>embedding_hid</td><td>{hid}</td></tr>"
            f"<tr><td>train_iters</td><td>{iters}</td></tr></table>"
            f"<p class='meta'>{gcs_note}</p>"
            f"<p class='meta'>params_source='{params_source}': 'auto' reads hpo/best/latest.json "
            f"from GCS when it exists (falls back to config); 'config' forces config values.</p>")
        return HP(lr, hid, iters, source)

    # ---------- tuning ----------
    from google.cloud import aiplatform
    from google.cloud import storage
    from google.cloud.aiplatform import hyperparameter_tuning as hpt
    aiplatform.init(project=project, location=region, staging_bucket=f"gs://{bucket}")
    # GPU dla TRIALI (nie dla tego komponentu - on tylko czeka na wynik)
    machine_spec = {"machine_type": hpo_machine_type}
    if hpo_accelerator_type:
        machine_spec["accelerator_type"] = hpo_accelerator_type
        machine_spec["accelerator_count"] = hpo_accelerator_count
    print(f"TRIAL MACHINE SPEC: {machine_spec}")
    if "accelerator_type" in machine_spec:
        print("UWAGA: triale zadaja GPU - wymaga quoty 'Custom model training "
              "NVIDIA T4 GPUs' w Vertex Training (INNEJ niz quota Workbencha). "
              "Wiszace triale w Queued = brak quoty.")
    worker_pool_specs = [{
        "machine_spec": machine_spec,
        "replica_count": 1,
        "container_spec": {
            "image_uri": training_image,
            "command": ["python", "scripts/hpo_train_entry.py"],
            "args": ["--project", project, "--bucket", bucket, "--dataset", dataset,
                     "--seed", str(seed), "--n-way", str(n_way), "--k-shot", str(k_shot),
                     "--query", str(query), "--val-episodes", str(hpo_val_episodes)],
        },
    }]
    custom_job = aiplatform.CustomJob(display_name="fsl-hpo-trial",
                                      worker_pool_specs=worker_pool_specs)
    hpo_job = aiplatform.HyperparameterTuningJob(
        display_name="fsl-protonet-hpo",
        custom_job=custom_job,
        metric_spec={"val_accuracy": "maximize"},
        parameter_spec={
            "lr": hpt.DoubleParameterSpec(min=1e-4, max=1e-2, scale="log"),
            "embedding_hid": hpt.DiscreteParameterSpec(values=[32, 64, 128], scale="linear"),
            "train_iters": hpt.IntegerParameterSpec(min=200, max=1200, scale="linear"),
        },
        max_trial_count=hpo_max_trials,
        parallel_trial_count=hpo_parallel,
        max_failed_trial_count=5,   # pojedyncze porazki nie ubijaja strojenia
    )
    # BEZ timeout: empirycznie (3 przebiegi) scheduling.timeout obejmowal czas
    # w kolejce i ubijal joby czekajace na zasoby jako CANCELED. Hamulec
    # kosztu = max_trial_count; wiszace triale rozwiazuje sie quota, nie zegarem.
    hpo_job.run()

    rows = []
    best_val, best = -1.0, None
    for t in hpo_job.trials:
        if t.final_measurement and t.final_measurement.metrics:
            v = t.final_measurement.metrics[0].value
            params = {p.parameter_id: p.value for p in t.parameters}
            rows.append((v, t.id, params))
            if v > best_val:
                best_val, best = v, params
    assert best is not None, "HPO finished without any successful trial"
    rows.sort(reverse=True)
    print(f"HPO best: val_accuracy={best_val:.4f}, params={best}")

    # --- zapis zwyciezcy do GCS: latest + kopia z timestampem (append-only historia) ---
    saved_at = time.strftime("%Y%m%d-%H%M%S")
    record = {"lr": float(best["lr"]), "embedding_hid": int(best["embedding_hid"]),
              "train_iters": int(best["train_iters"]), "val_accuracy": float(best_val),
              "saved_at": saved_at, "hpo_job": hpo_job.resource_name,
              "dataset": dataset, "seed": seed}
    gcs_bucket = storage.Client(project=project).bucket(bucket)
    payload = json.dumps(record, indent=2)
    gcs_bucket.blob(f"hpo/best/{saved_at}.json").upload_from_string(payload)
    gcs_bucket.blob("hpo/best/latest.json").upload_from_string(payload)
    print(f"Best params saved to gs://{bucket}/hpo/best/latest.json (+ {saved_at}.json)")

    # --- raport HTML: ranking triali, zwyciezca wyrozniony ---
    lo = min(v for v, _, _ in rows)
    span = (best_val - lo) or 1e-9
    table = "<tr><th>#</th><th>trial</th><th>val_accuracy</th><th>lr</th><th>hid</th><th>iters</th><th></th></tr>"
    for rank, (v, tid, p) in enumerate(rows, 1):
        w = int(20 + 200 * (v - lo) / span)
        bar = (f'<svg width="230" height="14"><rect width="{w}" height="14" rx="3" '
               f'fill="{"#2a7d4f" if rank == 1 else "#b8c4d8"}"/></svg>')
        cls = ' class="win"' if rank == 1 else ""
        table += (f'<tr{cls}><td>{rank}</td><td>{tid}</td><td>{v:.4f}</td>'
                  f'<td>{float(p["lr"]):.5f}</td><td>{int(p["embedding_hid"])}</td>'
                  f'<td>{int(p["train_iters"])}</td><td>{bar}</td></tr>')
    write_report(
        f"<h2>HPO result - {len(rows)} completed trials</h2>"
        f"<p class='src'>winner: val_accuracy={best_val:.4f} | lr={float(best['lr']):.5f}, "
        f"hid={int(best['embedding_hid'])}, iters={int(best['train_iters'])}</p>"
        f"<table>{table}</table>"
        f"<p class='meta'>Saved to gs://{bucket}/hpo/best/latest.json (+ timestamped copy). "
        f"Future passthrough runs (hpo.enabled=false, params_source='auto') will use it "
        f"automatically. Tuned on VALIDATION; the gates downstream judge on TEST.</p>"
        f"<p class='meta'>job: {hpo_job.resource_name}</p>")
    return HP(float(best["lr"]), int(best["embedding_hid"]), int(best["train_iters"]), "hpo")

## Component 4: `heavy_eval_container`

In [6]:
# --- CIEZKA EWALUACJA (kontener): laduje ZAPISANY artefakt, przelicza niezaleznie ---
@dsl.container_component
def heavy_eval_container(
    project: str,
    region: str,
    bucket: str,
    dataset: str,
    model_dir: str,
    model_version: str,
    train_accuracy_mean: float,
    train_accuracy_ci95: float,
    accuracy_threshold: float,
    max_std_threshold: float,
    test_episodes: int,
    passed: OutputPath(bool),
    reason: OutputPath(str),
    accuracy_recomputed: OutputPath(float),
    report: Output[HTML],
):
    return ContainerSpec(
        image=IMAGE,
        command=["python", "scripts/evaluate_pipeline_entry.py"],
        args=[
            "--project", project, "--region", region, "--bucket", bucket,
            "--dataset", dataset, "--model-dir", model_dir,
            "--model-version", model_version,
            "--train-accuracy-mean", train_accuracy_mean,
            "--train-accuracy-ci95", train_accuracy_ci95,
            "--accuracy-threshold", accuracy_threshold,
            "--max-std-threshold", max_std_threshold,
            "--test-episodes", test_episodes,
            "--passed-output-path", passed,
            "--reason-output-path", reason,
            "--accuracy-recomputed-output-path", accuracy_recomputed,
            "--report-html-path", report.path,
        ],
    )

## Component 5: `explain_container`

In [7]:
# --- EXPLAINABILITY (kontener): raport case-based reasoning, always-on po heavy ---
@dsl.container_component
def explain_container(
    project: str,
    region: str,
    bucket: str,
    dataset: str,
    model_dir: str,
    model_version: str,
    n_episodes: int,
    report: Output[HTML],
    summary: OutputPath(str),
):
    return ContainerSpec(
        image=IMAGE,
        command=["python", "scripts/explain_pipeline_entry.py"],
        args=[
            "--project", project, "--region", region, "--bucket", bucket,
            "--dataset", dataset, "--model-dir", model_dir,
            "--model-version", model_version,
            "--n-episodes", n_episodes,
            "--report-html-path", report.path,
            "--summary-output-path", summary,
        ],
    )

## Wrap training into a Custom Job

In [8]:
from google_cloud_pipeline_components.v1.custom_job import create_custom_training_job_op_from_component

# Trening = Custom Job (dedykowana maszyna, widoczny w Vertex Training),
# ktorego workerem jest NASZ kontener z fsl. Kontener + job naraz.
train_as_job = create_custom_training_job_op_from_component(
    train_container,
    display_name="fsl-training-job",
    machine_type=CFG["compute"]["machine_type"],
    replica_count=1,
)

## Component 6: `evaluate_gate_component`

In [9]:
@dsl.component(base_image="python:3.10")
def evaluate_gate_component(
    accuracy_mean: float, accuracy_ci95: float, accuracy_std: float,
    accuracy_threshold: float, max_std_threshold: float,
    gate_metrics: Output[Metrics],
) -> NamedTuple("G", [("passed", bool), ("reason", str)]):
    checks = []
    lower = accuracy_mean - accuracy_ci95
    checks.append(("accuracy_lower_ci", lower >= accuracy_threshold,
                   f"lowerCI={lower:.4f} vs threshold={accuracy_threshold:.4f}"))
    checks.append(("stability", accuracy_std <= max_std_threshold,
                   f"std={accuracy_std:.4f} vs max={max_std_threshold:.4f}"))
    passed = all(o for _, o, _ in checks)
    reason = "; ".join(f"{n}: {'PASS' if o else 'FAIL'} ({d})" for n, o, d in checks)
    gate_metrics.log_metric("gate_passed", 1.0 if passed else 0.0)
    gate_metrics.log_metric("accuracy_lower_ci", lower)
    gate_metrics.log_metric("margin_above_threshold", lower - accuracy_threshold)
    gate_metrics.log_metric("std_headroom", max_std_threshold - accuracy_std)
    print(f"Gate: {'PASSED' if passed else 'FAILED'} - {reason}")
    from collections import namedtuple
    return namedtuple("G", ["passed", "reason"])(passed, reason)

## Component 7: `register_model_component`

In [10]:
@dsl.component(base_image="python:3.10", packages_to_install=["google-cloud-aiplatform"])
def register_model_component(
    project: str, region: str, model_display_name: str,
    serving_container_image_uri: str, model_dir: str,
    accuracy_mean: float, data_sha: str, seed: int,
) -> NamedTuple("Reg", [("resource_name", str), ("version_id", str)]):
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region)
    existing = aiplatform.Model.list(filter=f'display_name="{model_display_name}"')
    parent = existing[0].resource_name if existing else None
    m = aiplatform.Model.upload(
        display_name=model_display_name, artifact_uri=model_dir,
        serving_container_image_uri=serving_container_image_uri, parent_model=parent,
        labels={"framework": "pytorch", "model": "protonet", "seed": str(seed)},
        description=f"ProtoNet acc={accuracy_mean:.4f} sha={data_sha[:12]}",
    )
    print(f"Registered {m.resource_name} v{m.version_id}")
    from collections import namedtuple
    return namedtuple("Reg", ["resource_name", "version_id"])(m.resource_name, str(m.version_id))

## Component 8: `log_experiment_component` (now logs lr + hyperparameter_source)

In [11]:
@dsl.component(base_image="python:3.10", packages_to_install=["google-cloud-aiplatform"])
def log_experiment_component(
    project: str, region: str, experiment_name: str, run_type: str, dataset: str,
    n_way: int, k_shot: int, query: int, seed: int, train_iters: int,
    lr: float, hp_source: str,
    data_sha: str, accuracy_mean: float, accuracy_ci95: float,
    accuracy_std: float, train_time: float, gate_passed: bool, gate_reason: str,
) -> str:
    import time
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region, experiment=experiment_name)
    run_name = f"pipeline-{run_type.replace('_','-')}-{n_way}w{k_shot}s-seed{seed}-{int(time.time())}"
    aiplatform.start_run(run_name)
    aiplatform.log_params({"model": "protonet", "n_way": n_way, "k_shot": k_shot,
        "query": query, "seed": seed, "train_iters": train_iters, "dataset": dataset,
        "lr": lr, "hyperparameter_source": hp_source,
        "dataset_archive_sha256": data_sha, "run_type": run_type,
        "gate_passed": gate_passed, "gate_reason": gate_reason})
    aiplatform.log_metrics({"test_accuracy_mean": accuracy_mean,
        "test_accuracy_ci95": accuracy_ci95, "test_accuracy_std": accuracy_std,
        "train_time_seconds": train_time})
    aiplatform.end_run()
    print(f"Logged run {run_name}")
    return run_name

## Component 9: `rejection_report_component`

In [12]:
# --- RAPORT ODRZUCENIA (lekki): HTML wyjasniajacy, dlaczego bramka nie puscila ---
@dsl.component(base_image="python:3.10")
def rejection_report_component(
    accuracy_mean: float, accuracy_ci95: float, accuracy_std: float,
    accuracy_threshold: float, max_std_threshold: float, gate_reason: str,
    report: Output[HTML],
) -> str:
    def svg_gauge(value, threshold, lo, hi, label, value_label, threshold_label,
                  width=640, height=90, pass_side="right"):
        pad = 60
        bar_w = width - 2 * pad
        y = 42

        def x_px(v):
            v = max(lo, min(hi, v))
            return pad + (v - lo) / (hi - lo or 1) * bar_w

        tx, vx = x_px(threshold), x_px(value)
        if pass_side == "right":
            fail_rect = f'<rect x="{pad}" y="{y}" width="{tx-pad:.1f}" height="14" fill="#f6d5d1"/>'
            pass_rect = f'<rect x="{tx:.1f}" y="{y}" width="{pad+bar_w-tx:.1f}" height="14" fill="#d9ead9"/>'
        else:
            pass_rect = f'<rect x="{pad}" y="{y}" width="{tx-pad:.1f}" height="14" fill="#d9ead9"/>'
            fail_rect = f'<rect x="{tx:.1f}" y="{y}" width="{pad+bar_w-tx:.1f}" height="14" fill="#f6d5d1"/>'
        ok = (value >= threshold) if pass_side == "right" else (value <= threshold)
        col = "#2a7d4f" if ok else "#c0392b"
        return (f'<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">'
                f'<text x="{pad}" y="20" font-size="13" fill="#333" font-weight="600">{label}</text>'
                f'{fail_rect}{pass_rect}'
                f'<line x1="{tx:.1f}" y1="{y-6}" x2="{tx:.1f}" y2="{y+20}" stroke="#c0392b" stroke-width="2"/>'
                f'<text x="{tx:.1f}" y="{y+34}" font-size="11" fill="#c0392b" text-anchor="middle">{threshold_label}</text>'
                f'<circle cx="{vx:.1f}" cy="{y+7}" r="7" fill="{col}"/>'
                f'<text x="{vx:.1f}" y="{y-10}" font-size="11" fill="{col}" text-anchor="middle" font-weight="600">{value_label}</text>'
                f'</svg>')

    lower = accuracy_mean - accuracy_ci95
    g1 = svg_gauge(lower, accuracy_threshold, min(0.5, lower - 0.05), 1.0,
                   "Criterion 1: lower 95% CI bound (must be RIGHT of the threshold)",
                   f"{lower:.3f}", f"threshold {accuracy_threshold}", pass_side="right")
    g2 = svg_gauge(accuracy_std, max_std_threshold, 0.0, max(0.15, accuracy_std * 1.5),
                   "Criterion 2: std / spread (must be LEFT of the max)",
                   f"{accuracy_std:.3f}", f"max {max_std_threshold}", pass_side="left")
    html = f"""<!DOCTYPE html><html><head><style>
body{{font-family:-apple-system,sans-serif;margin:24px;color:#1a1a1a;max-width:760px}}
.verdict{{font-size:18px;font-weight:700;color:#c0392b;margin:10px 0}}
.metric{{display:inline-block;margin:8px 28px 8px 0}}.metric .v{{font-size:24px;font-weight:600;color:#4C72B0}}
.metric .l{{font-size:12px;color:#666}}.reason{{font-size:13px;color:#555;background:#f7f7f7;padding:10px;border-radius:6px}}
.meta{{font-size:12px;color:#888}}</style></head><body>
<h2>Gate Rejection Report</h2>
<div class="verdict">MODEL REJECTED - not registered, not promoted</div>
<div class="metric"><div class="v">{accuracy_mean:.2%}</div><div class="l">mean accuracy</div></div>
<div class="metric"><div class="v">{lower:.4f}</div><div class="l">lower 95% CI</div></div>
<div class="metric"><div class="v">{accuracy_std:.4f}</div><div class="l">std (spread)</div></div>
{g1}{g2}
<p class="reason">{gate_reason}</p>
<p class="meta">A dot in the red zone is the reason for rejection. The mean alone can look fine while
the lower-CI bound (confidence, penalized by spread) or the spread itself fails - see the two gauges.</p>
</body></html>"""
    with open(report.path, "w") as f:
        f.write(html)
    print(f"Rejection report written. Reason: {gate_reason}")
    return "rejected-report-written"

## Components 10+11: promotion

In [13]:
# --- PROMOCJA (lekkie): alias "production" na wersji + retag obrazu ---
@dsl.component(base_image="python:3.10", packages_to_install=["google-cloud-aiplatform"])
def promote_model_component(
    project: str, region: str, model_resource_name: str, version_id: str,
) -> str:
    from google.cloud import aiplatform
    from google.cloud.aiplatform.models import ModelRegistry
    aiplatform.init(project=project, location=region)
    registry = ModelRegistry(model=model_resource_name, project=project, location=region)
    registry.add_version_aliases(["production"], version=version_id)
    print(f"Alias 'production' -> version {version_id} of {model_resource_name}")
    return f"{model_resource_name}@{version_id}"


@dsl.component(base_image="python:3.10", packages_to_install=["google-cloud-artifact-registry"])
def promote_image_component(
    project: str, region: str, repo: str, image_name: str,
    source_tag: str, target_tag: str,
) -> str:
    # Retag bez pobierania obrazu: tag docelowy wskazuje na TE SAMA wersje
    # (digest), na ktora wskazuje tag zrodlowy. Testowany artefakt == promowany.
    from google.api_core.exceptions import AlreadyExists, NotFound
    from google.cloud import artifactregistry_v1
    client = artifactregistry_v1.ArtifactRegistryClient()
    base = f"projects/{project}/locations/{region}/repositories/{repo}/packages/{image_name}"
    src = client.get_tag(name=f"{base}/tags/{source_tag}")
    print(f"Source {source_tag} -> version: {src.version}")
    tag_obj = artifactregistry_v1.Tag(name=f"{base}/tags/{target_tag}", version=src.version)
    try:
        client.create_tag(parent=base, tag=tag_obj, tag_id=target_tag)
        print(f"Created tag {target_tag}")
    except AlreadyExists:
        client.update_tag(tag=tag_obj)
        print(f"Updated tag {target_tag} to {src.version}")
    return f"{image_name}:{target_tag} -> {src.version.split('/')[-1][:24]}"

## Pipeline: verify -> resolve -> train -> gates -> endings

In [14]:
@dsl.pipeline(name="fsl-hello-pipeline",
              description="verify -> train[CustomJob+container] -> gate -> If(register)")
def training_pipeline(
    project: str, bucket: str, dataset: str = "omniglot", region: str = "us-central1",
    n_way: int = 5, k_shot: int = 5, query: int = 15, seed: int = 0,
    train_iters: int = 300, embedding_hid: int = 64, test_episodes: int = 1000,
    model_display_name: str = "fsl-protonet-omniglot",
    serving_container_image_uri: str = "us-central1-docker.pkg.dev/x/fsl-images/train:candidate",
    accuracy_threshold: float = 0.90, max_std_threshold: float = 0.05,
    image_repo: str = "fsl-images", image_name: str = "train",
    explain_episodes: int = 60,
    enable_hpo: bool = False,
    training_image: str = "",
    hpo_machine_type: str = "n1-standard-8",
    hpo_max_trials: int = 12, hpo_parallel: int = 3, hpo_val_episodes: int = 400,
    hpo_accelerator_type: str = "", hpo_accelerator_count: int = 0,
    hpo_params_source: str = "auto",
    lr: float = 0.001,
    run_type: str = "inner_loop", experiment_name: str = "fsl-inner-loop",
):
    verify_task = verify_frozen_component(project=project, bucket=bucket, dataset=dataset)

    # HPO albo passthrough - jeden komponent, flaga decyduje (Python-if w srodku)
    resolve_task = resolve_hyperparameters(
        project=project, region=region, bucket=bucket, dataset=dataset,
        training_image=training_image, hpo_machine_type=hpo_machine_type,
        hpo_accelerator_type=hpo_accelerator_type,
        hpo_accelerator_count=hpo_accelerator_count,
        params_source=hpo_params_source,
        enable_hpo=enable_hpo,
        default_lr=lr, default_embedding_hid=embedding_hid, default_train_iters=train_iters,
        hpo_max_trials=hpo_max_trials, hpo_parallel=hpo_parallel,
        hpo_val_episodes=hpo_val_episodes,
        seed=seed, n_way=n_way, k_shot=k_shot, query=query,
    ).after(verify_task)

    train_task = train_as_job(
        project=project, location=region,
        bucket=bucket, dataset=dataset, n_way=n_way, k_shot=k_shot, query=query,
        seed=seed,
        train_iters=resolve_task.outputs["train_iters"],
        embedding_hid=resolve_task.outputs["embedding_hid"],
        lr=resolve_task.outputs["lr"],
    )

    gate_task = evaluate_gate_component(
        accuracy_mean=train_task.outputs["accuracy_mean"],
        accuracy_ci95=train_task.outputs["accuracy_ci95"],
        accuracy_std=train_task.outputs["accuracy_std"],
        accuracy_threshold=accuracy_threshold, max_std_threshold=max_std_threshold,
    )

    with dsl.If(gate_task.outputs["passed"] == True, name="gate-passed"):
        register_task = register_model_component(
            project=project, region=region, model_display_name=model_display_name,
            serving_container_image_uri=serving_container_image_uri,
            model_dir=train_task.outputs["model_dir"],
            accuracy_mean=train_task.outputs["accuracy_mean"],
            data_sha=train_task.outputs["data_sha"], seed=seed,
        )

        # CIEZKA EWALUACJA zarejestrowanej wersji: laduje artefakt z GCS,
        # przelicza niezaleznie, testuje spojnosc, renderuje raport HTML.
        heavy_task = heavy_eval_container(
            project=project, region=region, bucket=bucket, dataset=dataset,
            model_dir=train_task.outputs["model_dir"],
            model_version=register_task.outputs["version_id"],
            train_accuracy_mean=train_task.outputs["accuracy_mean"],
            train_accuracy_ci95=train_task.outputs["accuracy_ci95"],
            accuracy_threshold=accuracy_threshold,
            max_std_threshold=max_std_threshold,
            test_episodes=test_episodes,
        )

        # EXPLAINABILITY: zawsze po heavy (niezaleznie od jej werdyktu) - opisuje
        # model raportem case-based reasoning; nie decyduje o niczym
        explain_container(
            project=project, region=region, bucket=bucket, dataset=dataset,
            model_dir=train_task.outputs["model_dir"],
            model_version=register_task.outputs["version_id"],
            n_episodes=explain_episodes,
        ).after(heavy_task)

        # DRUGA BRAMKA: promocja tylko gdy zapisany artefakt zweryfikowany
        with dsl.If(heavy_task.outputs["passed"] == True, name="heavy-passed"):
            promote_model_component(
                project=project, region=region,
                model_resource_name=register_task.outputs["resource_name"],
                version_id=register_task.outputs["version_id"],
            )
            promote_image_component(
                project=project, region=region,
                repo=image_repo, image_name=image_name,
                source_tag="candidate", target_tag="production",
            )

    # LOGOWANIE: bezwarunkowe (poza If) - loguje kazdy przebieg, tez odrzucony
    with dsl.Else(name="gate-failed"):
        rejection_report_component(
            accuracy_mean=train_task.outputs["accuracy_mean"],
            accuracy_ci95=train_task.outputs["accuracy_ci95"],
            accuracy_std=train_task.outputs["accuracy_std"],
            accuracy_threshold=accuracy_threshold,
            max_std_threshold=max_std_threshold,
            gate_reason=gate_task.outputs["reason"],
        )

        # EXPLAINABILITY takze przy odrzuceniu - diagnoza "jak ta slabosc
        # wyglada w srodku": zlewajace sie klastry, marginesy przesuniete
        # w lewo, przypadki bledne. Model jest w GCS niezaleznie od bramki.
        explain_container(
            project=project, region=region, bucket=bucket, dataset=dataset,
            model_dir=train_task.outputs["model_dir"],
            model_version="not-registered",
            n_episodes=explain_episodes,
        )

    log_experiment_component(
        project=project, region=region, experiment_name=experiment_name, run_type=run_type,
        dataset=dataset, n_way=n_way, k_shot=k_shot, query=query, seed=seed,
        train_iters=resolve_task.outputs["train_iters"],
        lr=resolve_task.outputs["lr"], hp_source=resolve_task.outputs["source"],
        data_sha=train_task.outputs["data_sha"],
        accuracy_mean=train_task.outputs["accuracy_mean"],
        accuracy_ci95=train_task.outputs["accuracy_ci95"],
        accuracy_std=train_task.outputs["accuracy_std"],
        train_time=train_task.outputs["train_time"],
        gate_passed=gate_task.outputs["passed"],
        gate_reason=gate_task.outputs["reason"],
    )

## Compile and submit

In [15]:
from kfp import compiler
from google.cloud import aiplatform

compiler.Compiler().compile(training_pipeline, PIPELINE_YAML)
print("Compiled ->", PIPELINE_YAML)

aiplatform.init(project=PROJECT, location=REGION)
parameter_values = {
    "project": PROJECT, "bucket": BUCKET, "dataset": DATASET, "region": REGION,
    "model_display_name": CFG["model_display_name"],
    "serving_container_image_uri": CFG["serving_container_image_uri"],
    "training_image": CFG["training_image_uri"],
    "accuracy_threshold": CFG["evaluation"]["accuracy_threshold"],
    "max_std_threshold": CFG["evaluation"]["max_std_threshold"],
    "run_type": CFG["run_type"], "experiment_name": CFG["experiment_name"],
    "image_repo": "fsl-images", "image_name": "train",
    "explain_episodes": 60,
    # --- HPO mode: the flag from config ---
    "enable_hpo": CFG["hpo"]["enabled"],
    "hpo_machine_type": CFG["hpo"]["machine_type"],
    "hpo_max_trials": CFG["hpo"]["max_trials"],
    "hpo_parallel": CFG["hpo"]["parallel"],
    "hpo_accelerator_type": CFG["hpo"].get("accelerator_type", ""),
    "hpo_accelerator_count": CFG["hpo"].get("accelerator_count", 0),
    "hpo_params_source": CFG["hpo"].get("params_source", "auto"),
    "hpo_val_episodes": CFG["hpo"]["val_episodes"],
    "lr": CFG["training"]["lr"],
    **{k: CFG["training"][k] for k in ["n_way","k_shot","query","seed","train_iters","embedding_hid","test_episodes"]},
}
job = aiplatform.PipelineJob(
    display_name="fsl-selftuning-pipeline",
    template_path=PIPELINE_YAML,
    pipeline_root=f"gs://{BUCKET}/pipeline-root",
    parameter_values=parameter_values,
)
job.submit()
mode = "HPO SEARCH (hours!)" if CFG["hpo"]["enabled"] else "passthrough (config values)"
print(f"Submitted in mode: {mode}")
print("resolve-hyperparameters logs will show either the passthrough or the Vizier progress;")
print("with HPO on, the tuning job also appears in Training -> Hyperparameter tuning jobs.")

/home/jupyter/envs/fsl/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/home/jupyter/envs/fsl/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/home/jupyter/envs/fsl/lib/python3.10/site-packages/

Compiled -> train_pipeline.yaml
Creating PipelineJob
PipelineJob created. Resource name: projects/815755318672/locations/us-central1/pipelineJobs/fsl-hello-pipeline-20260722114804
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/815755318672/locations/us-central1/pipelineJobs/fsl-hello-pipeline-20260722114804')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/fsl-hello-pipeline-20260722114804?project=815755318672
Submitted in mode: passthrough (config values)
resolve-hyperparameters logs will show either the passthrough or the Vizier progress;
with HPO on, the tuning job also appears in Training -> Hyperparameter tuning jobs.


## Publish template (auto-incremented)

In [16]:
import sys
sys.path.insert(0, "scripts")
from publish_template import publish_next_version

publish_next_version(project=PROJECT, region=REGION, repo=REPO,
    yaml_path=PIPELINE_YAML,
    description="Self-tuning pipeline: resolve-hyperparameters step (config flag: passthrough | Vizier)")

Existing version tags: ['latest', 'v1', 'v10', 'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v2', 'v20', 'v21', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9'] -> assigning v22
Published fsl-hello-pipeline @ v22 (version id: sha256:c705c14a9c837a1bb713013447802c905585874267d8328ddf21ccb046aa48ac)


('fsl-hello-pipeline',
 'sha256:c705c14a9c837a1bb713013447802c905585874267d8328ddf21ccb046aa48ac',
 'v22')

## Recommended first runs

1. **`hpo.enabled: false`** (as shipped) - confirms the refactor didn't break
   anything: same behavior as rung 8, but lr now explicitly flows and the log
   gains `hyperparameter_source: config`.
2. **`hpo.enabled: true`** - the real search (hours). Watch
   resolve-hyperparameters logs; afterwards, the winning values are visible in
   the log component's params AND you may want to copy them into the config as
   the new defaults for false-mode runs.

## What's next
The **production pipeline** - freezing the tuned recipe: same components,
`:production` image, weekly trigger, alias-move as deploy. The final rung.